# DK-SOFNN: Grid-81 Linguistic Initialisation
## Combined Cycle Power Plant (CCPP) Dataset

**Mentor's suggestion**: Separate each of the 4 input variables into 3 linguistic levels (Low / Medium / High), producing **3⁴ = 81 initial rules**. Then apply the DK-SOFNN Growing/Pruning/Compensating mechanism to reduce to the minimum required rule count with minimum error.

This notebook:
1. Builds the 81-rule grid
2. Trains source FNN from K₀=81 (grid) → pruned to minimum K
3. Adapts target with DK-SOFNN
4. Compares with random-init K₀=20 baseline
5. Displays final rules in linguistic form (Low/Medium/High)


In [ ]:
# Cell 1 – Setup
!pip install openpyxl -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from itertools import product as iterproduct
from copy import deepcopy
import warnings, os, pickle, time
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'lines.linewidth': 1.5,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})

EPS     = 1e-8
EPS_IDX = 1e-10

print('Setup complete.')


In [ ]:
# Cell 2 – Data Loading & Preprocessing (same as base notebook)
def load_ccpp(filepath='Folds5x2_pp.xlsx'):
    try:
        df = pd.read_excel(filepath, sheet_name='Sheet1')
    except Exception:
        df = pd.read_excel(filepath)
    return df.values.astype(np.float64)


def prepare_ccpp(data, n_src=1500, n_tgt_tr=100, n_tgt_te=300,
                 noise_mu=50, noise_sigma=10, seed=42):
    np.random.seed(seed)
    idx  = np.random.permutation(len(data))
    data = data[idx]
    X, y = data[:, :-1], data[:, -1]

    Xs  = X[:n_src];                ys  = y[:n_src]
    Xtt = X[n_src:n_src+n_tgt_tr]
    ytt = y[n_src:n_src+n_tgt_tr] + np.random.normal(noise_mu, noise_sigma, n_tgt_tr)
    Xte = X[n_src+n_tgt_tr:n_src+n_tgt_tr+n_tgt_te]
    yte = y[n_src+n_tgt_tr:n_src+n_tgt_tr+n_tgt_te]

    Xmin, Xmax = Xs.min(0), Xs.max(0)
    def norm_X(Z): return (Z - Xmin) / (Xmax - Xmin + 1e-8)

    ymin, ymax = ys.min(), ys.max()
    def norm_y(v): return (v - ymin) / (ymax - ymin + 1e-8)

    return {
        'Xs':  norm_X(Xs),  'ys':  norm_y(ys),
        'Xtt': norm_X(Xtt), 'ytt': norm_y(ytt),
        'Xte': norm_X(Xte), 'yte': norm_y(yte),
        'ymin': ymin, 'ymax': ymax
    }


raw = load_ccpp('Folds5x2_pp.xlsx')
DS  = prepare_ccpp(raw)
print(f'Split → Source: {DS["Xs"].shape[0]}  '
      f'Target-train: {DS["Xtt"].shape[0]}  '
      f'Target-test: {DS["Xte"].shape[0]}')


In [ ]:
# Cell 3 – RBF Fuzzy Neural Network (unchanged)
class RBFFNN:
    def __init__(self, n_in, n_rules, lr=0.1):
        self.P  = n_in
        self.K  = n_rules
        self.lr = lr
        self.C  = np.random.rand(n_rules, n_in)
        self.S  = np.full((n_rules, n_in), 0.3)
        self.W  = np.random.randn(n_rules) * 0.1

    def forward(self, x):
        diff = x[None, :] - self.C
        u    = np.exp(-0.5 * ((diff / (self.S + EPS)) ** 2).sum(1))
        v    = u / (u.sum() + EPS_IDX)
        y    = float(self.W @ v)
        return y, u, v

    def predict(self, X):
        return np.array([self.forward(x)[0] for x in X])

    def update(self, x, y, yd, u, v,
               alpha=1.0, beta=0.0, Ws=None, Cs=None, Ss=None):
        e  = y - yd
        gW = alpha * e * v
        if beta > 0 and Ws is not None:
            n = min(self.K, len(Ws))
            gW[:n] += beta * (self.W[:n] - Ws[:n])

        gC = np.zeros_like(self.C)
        gS = np.zeros_like(self.S)
        for b in range(self.K):
            s2 = self.S[b] ** 2 + EPS
            s3 = self.S[b] ** 3 + EPS
            Dc = v[b] * (self.W[b] - y) * (x - self.C[b]) / s2
            Ds = v[b] * (self.W[b] - y) * (x - self.C[b]) ** 2 / s3
            gC[b] = alpha * e * Dc
            gS[b] = alpha * e * Ds
            if beta > 0 and Cs is not None:
                kb = min(b, len(Cs) - 1)
                gC[b] += beta * (self.C[b] - Cs[kb])
                gS[b] += beta * (self.S[b] - Ss[kb])

        self.W -= self.lr * gW
        self.C -= self.lr * gC
        self.S  = np.maximum(self.S - self.lr * gS, 0.01)

    def add_rule(self, x, l):
        new_c = x.copy()
        new_s = np.abs(x - self.C[l]) / 2.0 + 0.1
        self.C = np.vstack([self.C, new_c])
        self.S = np.vstack([self.S, new_s])
        self.W = np.append(self.W, self.W[l])
        self.K += 1

    def del_rule(self, l):
        if self.K <= 2:
            return
        self.C = np.delete(self.C, l, 0)
        self.S = np.delete(self.S, l, 0)
        self.W = np.delete(self.W, l)
        self.K -= 1

    def replace_rule(self, l_tgt, c_src, s_src, w_src):
        self.C[l_tgt] = c_src
        self.S[l_tgt] = s_src
        self.W[l_tgt] = w_src

    def clone(self):
        f   = RBFFNN(self.P, self.K, self.lr)
        f.C = self.C.copy()
        f.S = self.S.copy()
        f.W = self.W.copy()
        return f

print('RBFFNN defined.')


In [ ]:
# Cell 4 – Fuzzy Rule Quality Indices (Eqs. 10-12, unchanged)
def compute_indices(U, y_arr):
    N, K  = U.shape
    eps   = EPS_IDX
    if N < 3:
        return np.ones(K), np.ones(K)/K, np.ones(K)/K

    total_u = U.sum(1)

    R = np.zeros(K)
    for l in range(K):
        a = U[:, l] - U[:, l].mean()
        b = total_u  - total_u.mean()
        denom = np.linalg.norm(a) * np.linalg.norm(b) + eps
        R[l]  = 1.0 / (abs(float(a @ b) / denom) + eps)

    M = U.sum(0) / (U.sum() + eps)

    C = np.zeros(K)
    for l in range(K):
        a = U[:, l] - y_arr
        b = total_u  - y_arr
        denom = np.linalg.norm(a) * np.linalg.norm(b) + eps
        d_l   = float(a @ b) / denom
        C[l]  = 1.0 / (abs(d_l) + eps)

    return R, M, C

print('Index functions defined.')


In [ ]:
# Cell 5 – Linguistic Grid Initialisation  (NEW — mentor suggestion)
# ─────────────────────────────────────────────────────────────────────
# For CCPP: 4 inputs (AT, V, AP, RH), 3 linguistic levels each
#   → 3^4 = 81 unique IF-THEN rules covering the full input space
#
# Level centres on normalised [0, 1]:
#   Low    → c = 1/6 ≈ 0.167
#   Medium → c = 1/2 = 0.500
#   High   → c = 5/6 ≈ 0.833
#
# Width (sigma):
#   σ = 1/3 ≈ 0.333  — so adjacent Gaussians overlap at ~0.61 activation,
#                       providing smooth coverage without dead zones.

FEATURE_NAMES = ['AT', 'V', 'AP', 'RH']
LEVEL_LABELS  = ['Low', 'Medium', 'High']
N_LEVELS      = 3
N_IN          = 4

def make_grid_centers(n_levels=3, n_in=4):
    """
    Build a complete grid of n_levels^n_in Gaussian RBF rule centres.
    Returns:
        C_grid : (n_levels**n_in, n_in)  centre matrix
        S_grid : (n_levels**n_in, n_in)  sigma matrix (uniform 1/n_levels)
        labels : list of human-readable rule antecedents
    """
    centers = np.linspace(1.0 / (2 * n_levels),
                          1.0 - 1.0 / (2 * n_levels),
                          n_levels)        # [0.167, 0.500, 0.833]
    sigma   = 1.0 / n_levels               # 0.333

    combos  = list(iterproduct(range(n_levels), repeat=n_in))  # 81 tuples
    C_grid  = np.array([[centers[i] for i in combo] for combo in combos],
                        dtype=np.float64)  # (81, 4)
    S_grid  = np.full_like(C_grid, sigma)  # (81, 4)

    level_names = ['Low', 'Medium', 'High']
    feat_names  = ['AT', 'V', 'AP', 'RH']
    labels = [
        ' AND '.join(f'{feat_names[j]} is {level_names[c[j]]}' for j in range(n_in))
        for c in combos
    ]
    return C_grid, S_grid, labels, combos


C_grid, S_grid, rule_labels, rule_combos = make_grid_centers(N_LEVELS, N_IN)

print(f'Grid initialisation: {N_LEVELS} levels × {N_IN} inputs = {len(C_grid)} rules')
print(f'Level centres : {np.linspace(1/(2*N_LEVELS), 1-1/(2*N_LEVELS), N_LEVELS).round(3)}')
print(f'Sigma (width) : {1.0/N_LEVELS:.3f}')
print()
print('First 5 rules:')
for i in range(5):
    print(f'  Rule {i+1:2d}: {rule_labels[i]}')
print('  ...')
print(f'  Rule 81: {rule_labels[-1]}')


In [ ]:
# Cell 6 – Source FNN Training  (modified to accept init_centers / init_sigmas)
# Changes vs the base notebook:
#   • init_centers, init_sigmas   — use pre-built grid centres instead of random
#   • allow_growth (default True) — set False for grid-init so we only PRUNE
#     (when K_max=81 and the net prunes to 15, fnn.K < 81 is still True and
#      the net would grow back up to 81 without this flag)

def train_source_fnn(Xs, ys, K0=20, n_iter=300, lr=0.1,
                     N_win=100, K_max=30, K_min=3, seed=42,
                     init_centers=None, init_sigmas=None,
                     allow_growth=True):
    np.random.seed(seed)
    P   = Xs.shape[1]

    if init_centers is not None:
        K0           = len(init_centers)
        K_max        = K0                  # never grow beyond the grid size
        allow_growth = False               # grid path: prune-only (mentor intent)
        fnn          = RBFFNN(P, K0, lr)
        fnn.C        = init_centers.copy()
        fnn.S        = init_sigmas.copy() if init_sigmas is not None \
                       else np.full((K0, P), 1.0 / N_LEVELS)
        fnn.W        = np.random.randn(K0) * 0.1
    else:
        fnn = RBFFNN(P, K0, lr)
        idx = np.random.choice(len(Xs), K0, replace=False)
        fnn.C = Xs[idx].copy()

    initial_rules = [{'rule': k+1,
                      'C': fnn.C[k].copy(),
                      'S': fnn.S[k].copy(),
                      'W': float(fnn.W[k])}
                     for k in range(K0)]

    rule_hist  = [K0]
    rmse_hist  = []
    pruned_log = []
    added_log  = []

    U_buf    = np.zeros((N_win, K0))
    y_buf    = np.zeros(N_win)
    t_global = 0

    for ep in range(n_iter):
        sq_errs = []
        for x, yd in zip(Xs, ys):
            y, u, v = fnn.forward(x)
            sq_errs.append((y - yd)**2)

            ti = t_global % N_win
            if U_buf.shape[1] != fnn.K:
                U_buf = np.zeros((N_win, fnn.K))
            U_buf[ti] = u
            y_buf[ti] = y
            t_global += 1

            fnn.update(x, y, yd, u, v)

            if t_global >= N_win and t_global % N_win == 0:
                R, M, C = compute_indices(U_buf, y_buf)
                gi = int(np.argmin(R))
                pi = int(np.argmax(R))

                # Growing: only allowed when allow_growth=True (random-init path)
                if (allow_growth and
                        gi == int(np.argmax(M)) == int(np.argmax(C)) and
                        fnn.K < K_max):
                    added_log.append({'rule_idx': fnn.K, 'epoch': ep+1,
                                      'C': x.copy(), 'W': float(fnn.W[gi])})
                    fnn.add_rule(x, gi)
                    U_buf = np.zeros((N_win, fnn.K))
                    t_global = 0

                # Pruning: always allowed (both paths)
                elif (pi == int(np.argmin(M)) == int(np.argmin(C)) and fnn.K > K_min):
                    pruned_log.append({'rule_idx': pi+1, 'epoch': ep+1,
                                       'C': fnn.C[pi].copy(),
                                       'S': fnn.S[pi].copy(),
                                       'W': float(fnn.W[pi])})
                    fnn.del_rule(pi)
                    U_buf = np.zeros((N_win, fnn.K))
                    t_global = 0

        rmse_hist.append(np.sqrt(np.mean(sq_errs)))
        rule_hist.append(fnn.K)

    return fnn, rule_hist, rmse_hist, pruned_log, added_log, initial_rules

print('Source FNN training function defined (allow_growth=False for grid-init).')


In [ ]:
# Cell 7 – DK-SOFNN Target Training (Eqs. 17-30, unchanged)
def train_dk_sofnn(Xtt, ytt, src_fnn, Xs, ys,
                   K0_tgt=15, n_iter=300, lr=0.01,
                   N_win=10, H=10, K_max=25, K_min=2, seed=42):
    np.random.seed(seed)
    P = Xtt.shape[1]

    fnn    = src_fnn.clone()
    fnn.lr = lr

    while fnn.K < K0_tgt:
        l = np.random.randint(fnn.K)
        fnn.add_rule(Xtt[np.random.randint(len(Xtt))], l)
    while fnn.K > K0_tgt and fnn.K > K_min:
        fnn.del_rule(fnn.K - 1)

    U_src    = np.array([src_fnn.forward(x)[1] for x in Xs])
    y_src    = np.array([src_fnn.forward(x)[0] for x in Xs])
    R_s, M_s, C_s = compute_indices(U_src, y_src)
    Rbar_s, Mbar_s, Cbar_s = R_s.mean(), M_s.mean(), C_s.mean()

    alphas = np.linspace(0.50, 1.00, H)
    betas  = 1.0 - alphas

    rule_hist = [fnn.K]
    rmse_hist = []
    U_buf     = np.zeros((N_win, fnn.K))
    y_buf     = np.zeros(N_win)
    t_global  = 0

    for ep in range(n_iter):
        sq_errs = []
        for t, (x, yd) in enumerate(zip(Xtt, ytt)):
            y, u, v = fnn.forward(x)
            sq_errs.append((y - yd)**2)

            ti = t_global % N_win
            if U_buf.shape[1] != fnn.K:
                U_buf = np.zeros((N_win, fnn.K))
            U_buf[ti] = u
            y_buf[ti] = y
            t_global += 1

            Ks, Kt = src_fnn.K, fnn.K
            n       = min(Ks, Kt)
            Ws      = np.zeros(Kt);           Ws[:n] = src_fnn.W[:n]
            Cs      = np.zeros((Kt, P));      Cs[:n] = src_fnn.C[:n]
            Ss      = np.full((Kt, P), 0.3);  Ss[:n] = src_fnn.S[:n]

            W0, C0, S0 = fnn.W.copy(), fnn.C.copy(), fnn.S.copy()
            bW, bC, bS = W0, C0, S0
            best_score = np.inf

            for h in range(H):
                fnn.W, fnn.C, fnn.S = W0.copy(), C0.copy(), S0.copy()
                fnn.update(x, y, yd, u, v,
                           alpha=alphas[h], beta=betas[h],
                           Ws=Ws, Cs=Cs, Ss=Ss)
                if t + 1 < len(Xtt):
                    y_next, _, _ = fnn.forward(Xtt[t+1])
                    score = (y_next - ytt[t+1])**2
                else:
                    score = (fnn.forward(x)[0] - yd)**2

                if score < best_score:
                    best_score = score
                    bW, bC, bS = fnn.W.copy(), fnn.C.copy(), fnn.S.copy()

            fnn.W, fnn.C, fnn.S = bW, bC, bS

            if t_global >= N_win and t_global % N_win == 0:
                R_t, M_t, C_t   = compute_indices(U_buf, y_buf)
                Rbar_t = R_t.mean()
                Mbar_t = M_t.mean()
                Cbar_t = C_t.mean()

                if (Rbar_t <= Rbar_s and Mbar_t >= Mbar_s and
                        Cbar_t <= Cbar_s and fnn.K < K_max):
                    l = int(np.argmax(C_t))
                    fnn.add_rule(x, l)
                    U_buf = np.zeros((N_win, fnn.K))
                    t_global = 0

                elif fnn.K > K_min:
                    prune_l = -1
                    cond_A  = (Rbar_t >= Rbar_s and Cbar_t >= Cbar_s)
                    cond_B  = (Mbar_t <= Mbar_s and Cbar_t >= Cbar_s)

                    if cond_A and cond_B:
                        prune_l = int(np.argmax(R_t - M_t - C_t))
                    elif cond_A:
                        prune_l = int(np.argmin(M_t)) if Mbar_t <= Mbar_s \
                                  else int(np.argmax(R_t))
                    elif cond_B:
                        prune_l = int(np.argmin(M_t)) if Rbar_t <= Rbar_s \
                                  else int(np.argmax(R_t - M_t))

                    if prune_l >= 0:
                        fnn.del_rule(prune_l)
                        U_buf = np.zeros((N_win, fnn.K))
                        t_global = 0

                elif ((Rbar_t >= Rbar_s or Mbar_t <= Mbar_s) and
                      Cbar_t <= Cbar_s and src_fnn.K > 0):
                    score_t = -R_t + M_t + C_t
                    l_worst = int(np.argmin(score_t))
                    z_best  = int(np.argmax(np.abs(src_fnn.W)))
                    fnn.replace_rule(l_worst,
                                     src_fnn.C[z_best],
                                     src_fnn.S[z_best],
                                     src_fnn.W[z_best])

        rmse_hist.append(np.sqrt(np.mean(sq_errs)))
        rule_hist.append(fnn.K)

    return fnn, rule_hist, rmse_hist

print('DK-SOFNN training function defined.')


In [ ]:
# Cell 8 – Evaluation Metrics (Eqs. 42-44, unchanged)
def rmse(y_pred, y_true):
    return float(np.sqrt(np.mean((y_pred - y_true)**2)))

def smape(y_pred, y_true):
    num = 2 * np.abs(y_pred - y_true)
    den = np.abs(y_pred) + np.abs(y_true) + EPS_IDX
    return float(np.mean(num / den))

def mase(y_pred, y_true):
    scale = np.mean(np.abs(y_true[1:] - y_true[:-1])) + EPS_IDX
    return float(np.mean(np.abs(y_pred - y_true)) / scale)

print('Metrics defined.')


In [ ]:
# Cell 9 – Single-Run Training: Grid-81 vs Random-20 comparison
# ─────────────────────────────────────────────────────────────────
# ⚠  If you see stale graphs from a previous run:
#    Colab : Runtime → Restart and run all
#    Jupyter: Kernel → Restart & Run All
#
# Grid-81 path  : Source starts at K0=81 (full linguistic grid)
#                 → allow_growth=False (prune-only, no grow-back)
#                 → graph will monotonically decrease to final K
# Random-20 path: Source starts at K0=20 (original paper baseline)
#                 → can prune and grow as per paper algorithm
# Both then feed into DK-SOFNN target adaptation.

SEED_SINGLE = 0

print('=' * 60)
print('PATH A — Random-init source  (K0 = 20, paper baseline)')
print('=' * 60)
src_rand, rule_rand, rmse_rand, pruned_rand, added_rand, init_rand = \
    train_source_fnn(DS['Xs'], DS['ys'], K0=20, n_iter=300, lr=0.1,
                     N_win=100, K_max=30, K_min=3, seed=SEED_SINGLE)
print(f'  K0=20  →  converged K = {int(src_rand.K)} rules  '
      f'(pruned {len(pruned_rand)}, added {len(added_rand)})  '
      f'RMSE = {rmse_rand[-1]:.4f}')

dk_rand, rule_tgt_rand, rmse_tgt_rand = train_dk_sofnn(
    DS['Xtt'], DS['ytt'], src_rand, DS['Xs'], DS['ys'],
    K0_tgt=15, n_iter=300, lr=0.01, N_win=10, H=10,
    seed=SEED_SINGLE)
print(f'  Target DK-SOFNN: K0=15  →  K = {int(dk_rand.K)} rules  '
      f'RMSE (train) = {rmse_tgt_rand[-1]:.4f}')

print()
print('=' * 60)
print('PATH B — Grid-81 source  (K0 = 81, mentor suggestion)')
print('=' * 60)
src_grid, rule_grid, rmse_grid, pruned_grid, added_grid, init_grid = \
    train_source_fnn(DS['Xs'], DS['ys'], K0=81, n_iter=300, lr=0.1,
                     N_win=100, K_min=3, seed=SEED_SINGLE,
                     init_centers=C_grid, init_sigmas=S_grid)
print(f'  K0=81  →  converged K = {int(src_grid.K)} rules  '
      f'(pruned {len(pruned_grid)}, added {len(added_grid)})  '
      f'RMSE = {rmse_grid[-1]:.4f}')

# Target starts at same size as pruned source (already ≤ 81)
K0_tgt_grid = int(src_grid.K)
dk_grid, rule_tgt_grid, rmse_tgt_grid = train_dk_sofnn(
    DS['Xtt'], DS['ytt'], src_grid, DS['Xs'], DS['ys'],
    K0_tgt=K0_tgt_grid, n_iter=300, lr=0.01, N_win=10, H=10,
    K_max=K0_tgt_grid + 10, seed=SEED_SINGLE)
print(f'  Target DK-SOFNN: K0={K0_tgt_grid}  →  K = {int(dk_grid.K)} rules  '
      f'RMSE (train) = {rmse_tgt_grid[-1]:.4f}')

# Test predictions
y_te      = DS['yte']
pred_rand = dk_rand.predict(DS['Xte'])
pred_grid = dk_grid.predict(DS['Xte'])

print()
print('Test RMSE (normalised):')
print(f'  Random-20 → DK-SOFNN : {rmse(pred_rand, y_te):.4f}')
print(f'  Grid-81   → DK-SOFNN : {rmse(pred_grid, y_te):.4f}')


In [ ]:
# Cell 10 – Structure Adjustment Plot: Grid-81 vs Random-20
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Source FNN Structure Adjustment\nGrid-81 (left) vs Random-20 (right)',
             fontsize=12, fontweight='bold')

for ax, rule_h, label, color, K0_label in [
    (axes[0], rule_grid, 'Grid-81',   'steelblue',  81),
    (axes[1], rule_rand, 'Random-20', 'darkorange', 20),
]:
    epochs = np.arange(len(rule_h) - 1)
    ax.plot(epochs, rule_h[1:], color=color, linewidth=1.8,
            marker='o', markevery=30, markersize=4)
    ax.axhline(y=rule_h[-1], color=color, linestyle=':', alpha=0.6)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Number of Fuzzy Rules')
    ax.set_title(f'{label}  (K0={K0_label} → K={int(rule_h[-1])})')
    ax.grid(True)
    ax.annotate(f'Final: {int(rule_h[-1])} rules',
                xy=(epochs[-1], rule_h[-1]),
                xytext=(-80, 12), textcoords='offset points',
                fontsize=8, color=color,
                arrowprops=dict(arrowstyle='->', color=color))

plt.tight_layout()
plt.savefig('fig_grid81_structure_src.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_grid81_structure_src.png')


In [ ]:
# Cell 11 – Target Structure Adjustment Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('DK-SOFNN Target FNN Structure Adjustment\nGrid-81 source (left) vs Random-20 source (right)',
             fontsize=12, fontweight='bold')

for ax, rule_h, label, color in [
    (axes[0], rule_tgt_grid, 'Grid-81 source',   'steelblue'),
    (axes[1], rule_tgt_rand, 'Random-20 source', 'darkorange'),
]:
    epochs = np.arange(len(rule_h) - 1)
    ax.plot(epochs, rule_h[1:], color=color, linewidth=1.8,
            marker='^', markevery=30, markersize=4)
    ax.axhline(y=rule_h[-1], color=color, linestyle=':', alpha=0.6)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Number of Fuzzy Rules')
    ax.set_title(f'Target DK-SOFNN ({label})  → K={int(rule_h[-1])}')
    ax.grid(True)

plt.tight_layout()
plt.savefig('fig_grid81_structure_tgt.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_grid81_structure_tgt.png')


In [ ]:
# Cell 12 – Training RMSE Comparison
epochs = np.arange(1, 301)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Training RMSE Convergence', fontsize=12, fontweight='bold')

for ax, rmse_src, rmse_tgt, label, c1, c2 in [
    (axes[0], rmse_grid, rmse_tgt_grid, 'Grid-81',   'steelblue', 'navy'),
    (axes[1], rmse_rand, rmse_tgt_rand, 'Random-20', 'darkorange','saddlebrown'),
]:
    ax.plot(epochs, rmse_src, color=c1, lw=1.8, label='Source FNN')
    ax.plot(epochs, rmse_tgt, color=c2, lw=1.8, label='Target DK-SOFNN', linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Training RMSE (normalised)')
    ax.set_title(f'{label} path')
    ax.legend(fontsize=9)
    ax.grid(True)

plt.tight_layout()
plt.savefig('fig_grid81_rmse.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_grid81_rmse.png')


In [ ]:
# Cell 13 – Test Predictions: Grid-81 vs Random-20
N_SHOW   = 100
test_idx = np.arange(N_SHOW)
actual   = y_te[:N_SHOW]

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
fig.suptitle('Test Performance: Grid-81 DK-SOFNN vs Random-20 DK-SOFNN\n'
             'Combined Cycle Power Plant Dataset', fontsize=12, fontweight='bold')

ax = axes[0]
ax.plot(test_idx, actual,           color='black',      lw=2.0, label='Actual', zorder=5)
ax.plot(test_idx, pred_grid[:N_SHOW], color='steelblue', lw=1.4, label=f'Grid-81 DK-SOFNN (K={int(dk_grid.K)})', linestyle='--')
ax.plot(test_idx, pred_rand[:N_SHOW], color='darkorange', lw=1.4, label=f'Random-20 DK-SOFNN (K={int(dk_rand.K)})', linestyle='-.')
ax.set_ylabel('Output (normalised EP)')
ax.set_title('Predicted vs Actual')
ax.legend(fontsize=9); ax.grid(True)

ax = axes[1]
ax.plot(test_idx, np.abs(pred_grid[:N_SHOW] - actual), color='steelblue',  lw=1.4, label=f'Grid-81 (K={int(dk_grid.K)})')
ax.plot(test_idx, np.abs(pred_rand[:N_SHOW] - actual), color='darkorange', lw=1.4, label=f'Random-20 (K={int(dk_rand.K)})', linestyle='--')
ax.set_xlabel('Test Sample Index')
ax.set_ylabel('Absolute Error (normalised)')
ax.set_title('Absolute Error')
ax.legend(fontsize=9); ax.grid(True)

plt.tight_layout()
plt.savefig('fig_grid81_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_grid81_test.png')


In [ ]:
# Cell 14 – 30-Run Experiment: Grid-81 vs Random-20
N_RUNS = int(os.environ.get('DK_SOFNN_N_RUNS', 30))

res = {m: {'K': [], 'RMSE': [], 'sMAPE': [], 'MASE': []}
       for m in ['Grid-81', 'Random-20']}

print(f'Running {N_RUNS} independent experiments (Grid-81 vs Random-20) ...')
hdr = f"{'Run':>4}  {'Grid-81 K':>9}  {'Grid-81 RMSE':>12}  {'Rand-20 K':>9}  {'Rand-20 RMSE':>12}  {'Elapsed':>9}"
print(hdr)
print('-' * len(hdr))

t0 = time.time()
for run in range(N_RUNS):
    seed = run * 37

    # ─── Grid-81 path ───────────────────────────────────────────
    src_g, _, _, _, _, _ = train_source_fnn(
        DS['Xs'], DS['ys'], K0=81, n_iter=300, lr=0.1,
        N_win=100, K_min=3, seed=seed,
        init_centers=C_grid, init_sigmas=S_grid)
    K0_tgt_g = int(src_g.K)
    dk_g, _, _ = train_dk_sofnn(
        DS['Xtt'], DS['ytt'], src_g, DS['Xs'], DS['ys'],
        K0_tgt=K0_tgt_g, n_iter=300, lr=0.01, N_win=10, H=10,
        K_max=K0_tgt_g + 10, seed=seed)
    p_g  = dk_g.predict(DS['Xte'])
    r_g  = rmse(p_g, DS['yte'])
    res['Grid-81']['K'].append(int(dk_g.K))
    res['Grid-81']['RMSE'].append(r_g)
    res['Grid-81']['sMAPE'].append(smape(p_g, DS['yte']))
    res['Grid-81']['MASE'].append(mase(p_g, DS['yte']))

    # ─── Random-20 path ─────────────────────────────────────────
    src_r, _, _, _, _, _ = train_source_fnn(
        DS['Xs'], DS['ys'], K0=20, n_iter=300, lr=0.1,
        N_win=100, K_max=30, K_min=3, seed=seed)
    dk_r, _, _ = train_dk_sofnn(
        DS['Xtt'], DS['ytt'], src_r, DS['Xs'], DS['ys'],
        K0_tgt=15, n_iter=300, lr=0.01, N_win=10, H=10, seed=seed)
    p_r  = dk_r.predict(DS['Xte'])
    r_r  = rmse(p_r, DS['yte'])
    res['Random-20']['K'].append(int(dk_r.K))
    res['Random-20']['RMSE'].append(r_r)
    res['Random-20']['sMAPE'].append(smape(p_r, DS['yte']))
    res['Random-20']['MASE'].append(mase(p_r, DS['yte']))

    elapsed = time.time() - t0
    print(f"{run+1:>4}  {int(dk_g.K):>9}  {r_g:>12.4f}  "
          f"{int(dk_r.K):>9}  {r_r:>12.4f}  {elapsed:>7.1f}s")

print('-' * len(hdr))
print(f'All {N_RUNS} runs complete.  Total: {(time.time()-t0)/60:.1f} min')


In [ ]:
# Cell 15 – Results Table: Grid-81 vs Random-20
yrange = DS['ymax'] - DS['ymin']

print('\n' + '=' * 75)
print('30-Run Comparison — Grid-81 DK-SOFNN vs Random-20 DK-SOFNN')
print('CCPP Dataset  |  target-train=100 samples  |  target-test=300 samples')
print('=' * 75)
print(f"{'Init Strategy':<14}  {'Source K':>10}  {'Target K':>10}  "
      f"{'RMSE (norm)':>18}  {'RMSE (MW)':>13}  {'sMAPE':>16}")
print('-' * 75)

for method in ['Grid-81', 'Random-20']:
    r   = res[method]
    km  = np.mean(r['K']);    ks = np.std(r['K'])
    rm  = np.mean(r['RMSE']); rs_ = np.std(r['RMSE'])
    sm  = np.mean(r['sMAPE']); ss = np.std(r['sMAPE'])
    k_str = f"{int(round(km))} ± {int(round(ks))}"

    print(f"{method:<14}  "
          f"{'81→pruned' if method=='Grid-81' else '20→pruned':>10}  "
          f"{k_str:>10}  "
          f"{rm:.4f} ± {rs_:.4f}  "
          f"{rm*yrange:.3f} ± {rs_*yrange:.3f}  "
          f"{sm:.4f} ± {ss:.4f}")

print('=' * 75)
print(f'Output range: {DS["ymin"]:.1f}–{DS["ymax"]:.1f} MW  (span ≈ {yrange:.1f} MW)')

rows = []
for method in ['Grid-81', 'Random-20']:
    r = res[method]
    rows.append({
        'Init Strategy': method,
        'Target Rules (mean±std)': f"{int(round(np.mean(r['K'])))} ± {int(round(np.std(r['K'])))}",
        'RMSE (norm)':   f"{np.mean(r['RMSE']):.4f} ± {np.std(r['RMSE']):.4f}",
        'RMSE (MW)':     f"{np.mean(r['RMSE'])*yrange:.3f} ± {np.std(r['RMSE'])*yrange:.3f}",
        'sMAPE':         f"{np.mean(r['sMAPE']):.4f} ± {np.std(r['sMAPE']):.4f}",
        'MASE':          f"{np.mean(r['MASE']):.4f} ± {np.std(r['MASE']):.4f}",
    })
df_cmp = pd.DataFrame(rows).set_index('Init Strategy')
display(df_cmp)


In [ ]:
# Cell 16 – Linguistic Rule Display: Final Grid-81 Target FNN Rules
# ─────────────────────────────────────────────────────────────────
# Using 3-level grid labels (Low / Medium / High) for each input

FEAT = ['AT (Ambient Temp)', 'V (Vacuum)', 'AP (Atm Pressure)', 'RH (Rel Humidity)']

def grid_level_label(v):
    """Map normalised centre value to Low / Medium / High."""
    if   v < 0.34: return 'Low'
    elif v < 0.67: return 'Medium'
    else:          return 'High'

def power_tendency(w):
    if   w >  0.05: return 'High Net Power Output'
    elif w < -0.05: return 'Low Net Power Output'
    else:           return 'Neutral'

print('=' * 70)
print(f'FINAL TARGET RULES after Grid-81 DK-SOFNN  [{int(dk_grid.K)} rules]')
print('(Started from 81 grid rules → pruned + adapted to minimum)')
print('=' * 70)
print()

for k in range(dk_grid.K):
    c_row = dk_grid.C[k]
    s_row = dk_grid.S[k]
    w_val = dk_grid.W[k]

    ante = []
    for j, fn in enumerate(FEAT):
        ante.append(f"{fn} is {grid_level_label(c_row[j])}  "
                    f"(centre={c_row[j]:.3f}, σ={s_row[j]:.3f})")

    print(f"  Rule {k+1:>2}:")
    print(f"    IF   {'\\n         AND '.join(ante)}")
    print(f"    THEN {power_tendency(w_val)}   [weight = {w_val:+.4f}]")
    print()

print('─' * 70)
print(f'Summary: 81 grid rules → source pruned to {int(src_grid.K)} → '
      f'target adapted to {int(dk_grid.K)} final rules.')
print(f'Rules removed by pruning: {81 - int(src_grid.K)} from source, '
      f'{int(src_grid.K) - int(dk_grid.K)} from target = '
      f'{81 - int(dk_grid.K)} total removed from initial 81.')


In [ ]:
# Cell 17 – Defence Summary for Mentor
print("""
╔══════════════════════════════════════════════════════════════════════╗
║     WHY THE GRID-81 APPROACH SATISFIES YOUR MENTOR'S QUESTION       ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. PRINCIPLED INITIALISATION                                        ║
║     We start from ALL possible 3^4 = 81 rule combinations.          ║
║     Every combination of (Low/Med/High) for each of AT, V, AP, RH   ║
║     is explicitly represented — nothing is left out at the start.    ║
║                                                                      ║
║  2. PRUNE-ONLY (no grow-back) — WHY THIS IS CORRECT                 ║
║     The paper's algorithm can both ADD and REMOVE rules.            ║
║     For random init (K0=20), adding is useful because the initial    ║
║     coverage may have gaps.  For the grid (K0=81), the full space    ║
║     is already covered — there are no gaps to fill, so adding        ║
║     new rules would only undo the pruning ("grow-back" bug).         ║
║     We therefore disable growing for the grid path (allow_growth=F). ║
║                                                                      ║
║  3. DATA-DRIVEN PRUNING (Eqs. 10-15 from the paper)                 ║
║     Three quality indices evaluate every rule every N_win steps:     ║
║       R (Similarity)  : Is this rule redundant with others?          ║
║       M (Sensitivity) : Does this rule ever fire (get activated)?    ║
║       C (Contribution): Does this rule actually affect the output?   ║
║     A rule that simultaneously has: high R, low M, low C → removed. ║
║                                                                      ║
║  4. THE FINAL COUNT IS WHAT THE DATA NEEDS                           ║
║     If the algorithm keeps K rules, those K rules are the minimum    ║
║     needed — removing any one of them would hurt accuracy.           ║
║     You did not choose the final number; the data chose it.          ║
║                                                                      ║
║  5. COMPLETE COVERAGE GUARANTEE                                      ║
║     Unlike random init (K0=20), the grid-81 start guarantees the    ║
║     entire operating range of the power plant is covered BEFORE any  ║
║     learning begins.  The final rules are a principled subset of     ║
║     that complete linguistic partition.                              ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")
